<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Question_generation_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Train

In [1]:
!pip install transformers datasets evaluate accelerate rouge_score sacrebleu bert_score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.3 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,EarlyStoppingCallback
import evaluate

In [3]:
dataset=load_dataset('rajpurkar/squad')
dataset

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [4]:
dataset['train'][0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [5]:
def prepare_qg_example(example):
  context=example['context']
  question=example['question']
  answer=example['answers']['text'][0]

  input_text=(
      f"Generate a question based on the context.\n"
      f"Answer:{answer}\n"
      f"Context:{context}\n"
      f"Question:"
  )
  target_text=question

  return {
      'input_text':input_text,
      'target_text':target_text,
      'answer':answer,
      "context":context,
      "question":question
  }

In [6]:
qg_dataset=dataset.map(prepare_qg_example)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [7]:
qg_dataset['train'][0]['input_text']

'Generate a question based on the context.\nAnswer:Saint Bernadette Soubirous\nContext:Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.\nQuestion:'

In [8]:
qg_dataset['train'][0]['target_text']

'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'

In [9]:
train_data=qg_dataset['train'].shuffle(seed=42).select(range(10000))
val_data=qg_dataset['validation'].shuffle(seed=42).select(range(1000))

In [10]:
model_name='google/flan-t5-small'

tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
max_input_length=512
max_target_length=64

def tokenize_function(batch):
  model_inputs=tokenizer(batch['input_text'],padding=False,truncation=True,max_length=max_input_length)
  labels=tokenizer(text_target=batch['target_text'],padding=False,truncation=True,max_length=max_target_length)

  model_inputs['labels']=labels['input_ids']

  return model_inputs

In [12]:
tokenized_train=train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names
)
tokenized_val=val_data.map(
    tokenize_function,
    batched=True,
    remove_columns=val_data.column_names
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
data_collator=DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [14]:
bleu_metric=evaluate.load('sacrebleu')
rogue_metric=evaluate.load('rouge')

In [15]:
def compute_metric(eval_preds):
  preds,labels=eval_preds
  if isinstance(preds,tuple):
    preds=preds[0]
  if preds.ndim==3:
    preds=np.argmax(preds,axis=-1)
  preds=np.where(preds<0,tokenizer.pad_token_id,preds)
  labels=np.where(labels<0,tokenizer.pad_token_id,labels)

  preds=preds.astype(np.int64)
  labels=labels.astype(np.int64)

  decoded_preds=tokenizer.batch_decode(preds,skip_special_tokens=True)
  decoded_labels=tokenizer.batch_decode(labels,skip_special_tokens=True)

  decoded_preds=[pred.strip() for pred in decoded_preds]
  decoded_labels=[label.strip() for label in decoded_labels]

  bleu=bleu_metric.compute(predictions=decoded_preds,references=[[label] for label in decoded_labels])
  rogue=rogue_metric.compute(predictions=decoded_preds,references=decoded_labels)

  return{
      'bleu':bleu['score'],
      'rouge1':rogue['rouge1'],
      'rouge2':rogue['rouge2'],
      'rougeL':rogue['rougeL']
  }

In [16]:
training_args=Seq2SeqTrainingArguments(
    output_dir='./t5_question_generation',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    weight_decay=0.01,
    num_train_epochs=3,

    predict_with_generate=True,
    generation_max_length=64,

    fp16=False,
    save_total_limit=2,
    generation_num_beams=4,
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    report_to='none',
)

In [17]:
trainer=Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metric,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Rouge1,Rouge2,Rougel
1,3.918816,1.704798,17.020342,0.428476,0.219324,0.394853
2,3.762074,1.693895,17.262396,0.431618,0.221930,0.397539
3,3.537948,1.690531,16.947464,0.432986,0.223023,0.398766


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1875, training_loss=3.795425012207031, metrics={'train_runtime': 1156.9133, 'train_samples_per_second': 25.931, 'train_steps_per_second': 1.621, 'total_flos': 3488711665188864.0, 'train_loss': 3.795425012207031, 'epoch': 3.0})

##Eval

In [19]:
bertscore_metrics=evaluate.load('bertscore')


In [20]:
eval_results=trainer.evaluate()
eval_results

Training Loss,Validation Loss,Epoch,Bleu,Rouge1,Rouge2,Rougel
3.537948,1.690531,3,16.947464,0.432986,0.223023,0.398766


{'eval_loss': 1.6905310153961182,
 'eval_bleu': 16.947464419986048,
 'eval_rouge1': 0.4329862408040056,
 'eval_rouge2': 0.22302303037536286,
 'eval_rougeL': 0.3987658313127198}

In [21]:
def generate_question(context, answer):
    model.eval()

    input_text = (
        f"Generate a question based on the context.\n"
        f"Answer: {answer}\n"
        f"Context: {context}\n"
        f"Question:"
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    question = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return question

In [22]:
sample = qg_dataset["validation"][5]

context = sample["context"]
answer = sample["answer"]
reference_question = sample["question"]

generated_question = generate_question(context, answer)


print("Context:", context)
print("ANSWER:", answer)
print("REFERENCE:", reference_question)
print("GENERATED:", generated_question)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.
ANSWER: "golden anniversary"
REFERENCE: What was the theme of Super Bowl 50?
GENERATED: What was the name of Super Bowl 50?


In [27]:
def compute_single_metrics(prediction, reference):
    bleu = bleu_metric.compute(
        predictions=[prediction],
        references=[[reference]]
    )

    rouge = rogue_metric.compute(
        predictions=[prediction],
        references=[reference]
    )

    return {
        "bleu": bleu["score"],
        "rouge1": rouge["rouge1"],
        "rouge2": rouge["rouge2"],
        "rougeL": rouge["rougeL"]
    }

In [28]:
rows = []

predictions = []
references = []

for i in range(20):
    sample = qg_dataset["validation"][i]

    context = sample["context"]
    answer = sample["answer"]
    reference = sample["question"]

    prediction = generate_question(context, answer)
    scores = compute_single_metrics(prediction, reference)

    predictions.append(prediction)
    references.append(reference)

    rows.append({
        "context": context[:300] + "...",
        "answer": answer,
        "reference_question": reference,
        "generated_question": prediction,
        "bleu": scores["bleu"],
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"],
    })

bert_scores = bertscore_metrics.compute(
    predictions=predictions,
    references=references,
    model_type="distilbert-base-uncased"
)

for i in range(len(rows)):
    rows[i]["bertscore_precision"] = bert_scores["precision"][i]
    rows[i]["bertscore_recall"] = bert_scores["recall"][i]
    rows[i]["bertscore_f1"] = bert_scores["f1"][i]

eval_df = pd.DataFrame(rows)
eval_df

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,context,answer,reference_question,generated_question,bleu,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1
0,Super Bowl 50 was an American football game to...,Denver Broncos,Which NFL team represented the AFC at Super Bo...,Who defeated the Carolina Panthers 24–10?,4.410364,0.117647,0.000000,0.117647,0.733059,0.768986,0.750593
1,Super Bowl 50 was an American football game to...,Carolina Panthers,Which NFL team represented the NFC at Super Bo...,What team won the Super Bowl 50?,26.295890,0.588235,0.266667,0.588235,0.925941,0.879718,0.902238
2,Super Bowl 50 was an American football game to...,"Santa Clara, California",Where did Super Bowl 50 take place?,Where was the Super Bowl 50 game played?,20.164946,0.533333,0.307692,0.533333,0.910716,0.923274,0.916952
3,Super Bowl 50 was an American football game to...,Denver Broncos,Which NFL team won Super Bowl 50?,Who defeated the Carolina Panthers 24–10?,5.693025,0.000000,0.000000,0.000000,0.741242,0.796200,0.767739
4,Super Bowl 50 was an American football game to...,gold,What color was used to emphasize the 50th anni...,What was the name of Super Bowl 50?,7.483105,0.571429,0.105263,0.571429,0.873327,0.828609,0.850381
5,Super Bowl 50 was an American football game to...,"""golden anniversary""",What was the theme of Super Bowl 50?,What was the name of Super Bowl 50?,59.694918,0.875000,0.714286,0.875000,0.956451,0.956451,0.956451
6,Super Bowl 50 was an American football game to...,"February 7, 2016",What day was the game played on?,When was Super Bowl 50 played?,7.492443,0.307692,0.000000,0.307692,0.805907,0.825316,0.815496
7,Super Bowl 50 was an American football game to...,American Football Conference,What is the AFC short for?,Who defeated the Carolina Panthers 24–10 to ea...,3.377156,0.100000,0.000000,0.100000,0.615153,0.689898,0.650385
8,Super Bowl 50 was an American football game to...,"""golden anniversary""",What was the theme of Super Bowl 50?,What was the name of Super Bowl 50?,59.694918,0.875000,0.714286,0.875000,0.956451,0.956451,0.956451
9,Super Bowl 50 was an American football game to...,American Football Conference,What does AFC stand for?,Who defeated the Carolina Panthers 24–10 to ea...,2.839839,0.000000,0.000000,0.000000,0.586479,0.691556,0.634698
